# 🧪 FiberNet: Reinforcement & Validation / 强化与验证\n\n本教程在基础教学之后，深入展示：\n- **TPMS 超材料结构** (Triply Periodic Minimal Surfaces)\n- **FEM 有限元验证** (Gibson-Ashby 解析解基准)\n- **力学强化分析** (CPU-only, 无 GPU 依赖)\n- **收敛性研究** (h-refinement mesh sensitivity)

## 1. Setup / 环境设置

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import Normalize\nfrom matplotlib.cm import ScalarMappable\n\n# FiberNet imports\nfrom fibernet.core.fiber import Fiber\nfrom fibernet.core.network import FiberNetwork\nfrom fibernet.core.material import Material\nfrom fibernet.sim.mechanical import FiberFEM, stress_strain_curve\nfrom fibernet.sim.validation import (\n    gibson_ashby_honeycomb,\n    gibson_ashby_foam_3d,\n    validate_cantilever_beam,\n    validate_convergence,\n    validate_patch_test,\n    run_all_validations,\n    print_validation_report,\n)\n\nprint('✅ All imports successful')\nprint(f'NumPy: {np.__version__}')

---\n## 2. TPMS Structures / 三周期极小曲面结构\n\nTPMS (Triply Periodic Minimal Surfaces) 是一类具有零平均曲率的周期曲面，广泛用于轻质高强点阵结构设计。\n\nFiberNet 支持以下 TPMS 类型：\n- **Gyroid** (螺旋型)\n- **Diamond** (金刚石型)\n- **Primitive** (简单立方型)\n- **I-WP** (体心立方型)\n- **Neovius**\n- **Lidinoid**

In [ ]:
from fibernet.gen.tpms import tpms_sheet, tpms_lattice, tpms_gradient\n\n# ============================================================\n# TPMS Gallery\n\ntpms_types = ['gyroid', 'diamond', 'primitive', 'iwp']\n\nfig, axes = plt.subplots(2, 2, figsize=(14, 14))\naxes = axes.flatten()\n\nfor idx, tpms_type in enumerate(tpms_types):\n    ax = axes[idx]\n    \n    try:\n        net = tpms_lattice(\n            kind=tpms_type,\n            box_size=(10, 10, 10),\n            resolution=25,\n            num_periods=(1, 1, 1),\n            strut_radius=0.1,\n            seed=42,\n        )\n        \n        # Plot 2D projection\n        for fiber in net.fibers[:200]:\n            pts = fiber.centerline\n            ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.6, linewidth=0.5)\n        \n        ax.set_title(f'{tpms_type.upper()} ({len(net.fibers)} fibers)')\n        ax.set_xlabel('x')\n        ax.set_ylabel('y')\n        ax.set_aspect('equal')\n        ax.grid(True, alpha=0.3)\n        \n    except Exception as e:\n        ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center')\n        ax.set_title(tpms_type.upper())\n\nplt.tight_layout()\nplt.show()\nprint('TPMS structures generated successfully')

### 2.1 Gradient TPMS / 梯度 TPMS\n\n功能梯度 TPMS 结构在生物医学植入物和能量吸收器中有重要应用。

In [ ]:
# ============================================================\n# Gradient TPMS\n\ngrad_net = tpms_gradient(\n    kind='gyroid',\n    box_size=(20, 10, 10),\n    resolution=25,\n    gradient_axis=0,\n    gradient_range=(0.5, 2.0),\n    strut_radius=0.08,\n    seed=42,\n)\n\nfig, ax = plt.subplots(figsize=(12, 6))\n\nfor fiber in grad_net.fibers[:300]:\n    pts = fiber.centerline\n    ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.5, linewidth=0.4)\n\nax.set_title('Gradient Gyroid TPMS (unit cell size varies along x)')\nax.set_xlabel('x (gradient direction)')\nax.set_ylabel('y')\nax.set_aspect('equal')\nax.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()\nprint(f'Gradient TPMS: {len(grad_net.fibers)} fibers')

---\n## 3. FEM Validation / 有限元验证\n\n### 3.1 力学模拟技术说明\n\n**FiberNet 的力学模拟采用自研的有限元实现，不依赖任何开源 Python FEM 库**（如 FEniCS, SfePy, PyFEM 等）。\n\n核心实现：\n- **单元类型**: 3D Euler-Bernoulli 梁单元 (12 DOF/单元)\n- **刚度矩阵**: 解析推导的 12×12 局部刚度矩阵\n- **坐标变换**: 局部→全局变换矩阵 T\n- **稀疏求解**: scipy.sparse + spsolve (SuperLU)\n- **后处理**: 应变、应力、反力、应变能\n\n数学基础：\n$$K_{global} = T^T K_{local} T$$\n$$K_{local} = \begin{bmatrix} k_{axial} & 0 & 0 \\ 0 & k_{bending} & 0 \\ 0 & 0 & k_{torsion} \end{bmatrix}$$\n\n下面通过解析解验证实现的正确性。

In [ ]:
# ============================================================\n# Run all validation tests\n\nresults = run_all_validations(E_solid=1e9, verbose=True)\nprint()\nreport = print_validation_report(results)\nprint(report)

### 3.2 Cantilever Beam Validation / 悬臂梁验证\n\n悬臂梁是 FEM 最基本的验证案例：\n$$\delta_{tip} = \frac{PL^3}{3EI}$$

In [ ]:
# ============================================================\n# Detailed cantilever validation with convergence\n\nE_beam = 200e9  # Steel-like\nL_beam = 1.0\nr_beam = 0.01\nP_beam = 100.0\nI_beam = np.pi * r_beam**4 / 4\n\n# Analytical solution\ndelta_analytical = P_beam * L_beam**3 / (3 * E_beam * I_beam)\nprint(f'Analytical tip deflection: {delta_analytical:.6f} m')\nprint(f'EI = {E_beam * I_beam:.4e} N·m²')\nprint()\n\n# Convergence study\nsegment_counts = [2, 4, 8, 16, 32]\nerrors = []\n\nfor n_seg in segment_counts:\n    result = validate_cantilever_beam(\n        L=L_beam, E=E_beam, r=r_beam, P=P_beam,\n        segments=n_seg, tolerance=1.0\n    )\n    errors.append(result.relative_error)\n    print(f'  segments={n_seg:2d}: δ={result.numerical_value:.6f} m, error={result.relative_error:.4%}')\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: convergence\nax1.semilogy(segment_counts, errors, 'o-', linewidth=2, markersize=8)\nax1.set_xlabel('Number of segments')\nax1.set_ylabel('Relative error')\nax1.set_title('Cantilever Beam: Convergence')\nax1.grid(True, alpha=0.3)\nax1.axhline(y=0.01, color='r', linestyle='--', label='1% threshold')\nax1.legend()\n\n# Right: beam visualization\nx = np.linspace(0, L_beam, 100)\ny_analytical = (P_beam * x**2 / (6 * E_beam * I_beam)) * (3*L_beam - x)\nax2.plot(x, y_analytical * 1000, 'b-', linewidth=2, label='Analytical')\nax2.set_xlabel('x (m)')\nax2.set_ylabel('Deflection (mm)')\nax2.set_title('Cantilever Beam Deflection')\nax2.grid(True, alpha=0.3)\nax2.legend()\n\nplt.tight_layout()\nplt.show()

### 3.3 Gibson-Ashby Scaling / 蜂窝材料标度律\n\nGibson-Ashby 模型预测蜂窝材料的有效模量与相对密度的关系：\n$$\frac{E^*}{E_s} \propto \left(\frac{\rho^*}{\rho_s}\right)^n$$\n\n对于 2D 蜂窝：$n = 3$（弯曲主导）\n对于 3D 开孔泡沫：$n = 2$（拉伸主导）

In [ ]:
# ============================================================\n# Gibson-Ashby analytical scaling laws\n\nrho_ratios = np.linspace(0.01, 0.3, 50)\nE_solid = 1e9\n\n# 2D honeycomb (n=3)\nE_honeycomb = E_solid * (2/np.sqrt(3)) * rho_ratios**3\n\n# 3D open-cell foam (n=2)\nE_foam = E_solid * rho_ratios**2\n\n# 3D closed-cell foam\nphi = 0.8  # edge fraction\nE_closed = E_solid * (phi**2 * rho_ratios**2 + (1-phi) * rho_ratios)\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))\n\n# Linear scale\nax1.plot(rho_ratios, E_honeycomb/E_solid, 'b-', linewidth=2, label='2D Honeycomb (n=3)')\nax1.plot(rho_ratios, E_foam/E_solid, 'r-', linewidth=2, label='3D Open-cell (n=2)')\nax1.plot(rho_ratios, E_closed/E_solid, 'g-', linewidth=2, label='3D Closed-cell (φ=0.8)')\nax1.set_xlabel('Relative density ρ*/ρ_s')\nax1.set_ylabel('E*/E_s')\nax1.set_title('Gibson-Ashby Scaling Laws')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Log-log scale\nax2.loglog(rho_ratios, E_honeycomb/E_solid, 'b-', linewidth=2, label='2D Honeycomb (n=3)')\nax2.loglog(rho_ratios, E_foam/E_solid, 'r-', linewidth=2, label='3D Open-cell (n=2)')\nax2.loglog(rho_ratios, E_closed/E_solid, 'g-', linewidth=2, label='3D Closed-cell')\nax2.set_xlabel('Relative density ρ*/ρ_s')\nax2.set_ylabel('E*/E_s')\nax2.set_title('Gibson-Ashby Scaling (log-log)')\nax2.legend()\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

---\n## 4. Mechanical Reinforcement Analysis / 力学强化分析\n\n分析纤维网络在不同条件下的力学强化机制：\n- **密度效应**: 纤维数量对有效模量的影响\n- **取向效应**: 取向度对模量各向异性的影响\n- **交联效应**: 交联密度对刚度的影响

In [ ]:
from fibernet.gen.disordered import random_2d, oriented_2d\n\n# ============================================================\n# Density effect on effective modulus\n\nfiber_counts = [20, 40, 60, 80, 100, 150]\nE_effs_x = []\nE_effs_y = []\n\nE_solid = 1e9\nmaterial = Material(youngs_modulus=E_solid, poisson_ratio=0.3)\n\nprint('Density Effect Study (CPU):')\nprint('-' * 50)\n\nfor n_fibers in fiber_counts:\n    net = random_2d(\n        num_fibers=n_fibers,\n        fiber_length=5.0,\n        box_size=(10, 10),\n        fiber_radius=0.05,\n        material=material,\n        seed=42,\n    )\n    \n    fem = FiberFEM(net, segments_per_fiber=5)\n    \n    E_x = fem.effective_modulus(strain=0.001, axis=0)\n    E_y = fem.effective_modulus(strain=0.001, axis=1)\n    \n    E_effs_x.append(E_x)\n    E_effs_y.append(E_y)\n    \n    print(f'  N={n_fibers:3d}: E_x={E_x:.2e}, E_y={E_y:.2e} Pa')\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n\n# E vs fiber count\nax1.plot(fiber_counts, E_effs_x, 'o-', linewidth=2, label='E_x')\nax1.plot(fiber_counts, E_effs_y, 's-', linewidth=2, label='E_y')\nax1.set_xlabel('Number of fibers')\nax1.set_ylabel('Effective modulus (Pa)')\nax1.set_title('Density Effect')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Anisotropy ratio\naniso = [abs(x/y) if y > 0 else 0 for x, y in zip(E_effs_x, E_effs_y)]\nax2.plot(fiber_counts, aniso, 'o-', linewidth=2, color='g')\nax2.axhline(y=1.0, color='r', linestyle='--', label='Isotropic')\nax2.set_xlabel('Number of fibers')\nax2.set_ylabel('Anisotropy ratio |E_x/E_y|')\nax2.set_title('Anisotropy vs Density')\nax2.legend()\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

In [ ]:
# ============================================================\n# Orientation effect on anisotropy\n\nangles = [0, 15, 30, 45, 60, 75, 90]\nE_effs_oriented = []\norder_params = []\n\nprint('Orientation Effect Study (CPU):')\nprint('-' * 50)\n\nfor angle in angles:\n    try:\n        net = oriented_2d(\n            num_fibers=80,\n            fiber_length=5.0,\n            box_size=(10, 10),\n            orientation_angle=np.radians(angle),\n            orientation_spread=np.radians(10),\n            fiber_radius=0.05,\n            material=material,\n            seed=42,\n        )\n        \n        fem = FiberFEM(net, segments_per_fiber=5)\n        E_x = fem.effective_modulus(strain=0.001, axis=0)\n        E_y = fem.effective_modulus(strain=0.001, axis=1)\n        \n        E_effs_oriented.append((E_x, E_y))\n        \n        # Compute order parameter\n        nematic = 0\n        for f in net.fibers:\n            dx = f.centerline[-1, 0] - f.centerline[0, 0]\n            dy = f.centerline[-1, 1] - f.centerline[0, 1]\n            theta = np.arctan2(dy, dx)\n            nematic += np.cos(2 * theta)\n        nematic /= len(net.fibers)\n        order_params.append(nematic)\n        \n        print(f'  θ={angle:2d}°: E_x={E_x:.2e}, E_y={E_y:.2e}, S={nematic:.3f}')\n        \n    except Exception as e:\n        E_effs_oriented.append((0, 0))\n        order_params.append(0)\n        print(f'  θ={angle:2d}°: Error - {e}')\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n\n# E vs angle\nE_x_arr = [e[0] for e in E_effs_oriented]\nE_y_arr = [e[1] for e in E_effs_oriented]\nax1.plot(angles, E_x_arr, 'o-', linewidth=2, label='E_x')\nax1.plot(angles, E_y_arr, 's-', linewidth=2, label='E_y')\nax1.set_xlabel('Orientation angle (°)')\nax1.set_ylabel('Effective modulus (Pa)')\nax1.set_title('Orientation Effect')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Order parameter\nax2.plot(angles, order_params, 'o-', linewidth=2, color='purple')\nax2.set_xlabel('Orientation angle (°)')\nax2.set_ylabel('Nematic order parameter S')\nax2.set_title('Nematic Order vs Angle')\nax2.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

### 4.1 Stress-Strain Curves / 应力-应变曲线\n\n计算不同结构的应力-应变响应（增量加载，CPU 求解）。

In [ ]:
# ============================================================\n# Stress-strain curves for different network types\n\nfrom fibernet.gen.ordered import honeycomb_2d\n\nnetworks_to_test = {\n    'Random 2D': random_2d(\n        num_fibers=60, fiber_length=5.0, box_size=(10, 10),\n        fiber_radius=0.05, material=material, seed=42\n    ),\n    'Honeycomb': honeycomb_2d(\n        box_size=(10, 10), fiber_length=1.5,\n        fiber_radius=0.05, material=material\n    ),\n}\n\nfig, ax = plt.subplots(figsize=(10, 6))\ncolors = ['blue', 'red', 'green', 'orange']\n\nfor idx, (name, net) in enumerate(networks_to_test.items()):\n    strains, stresses = stress_strain_curve(\n        net, max_strain=0.05, num_steps=10, axis=0,\n        segments_per_fiber=5\n    )\n    \n    ax.plot(strains * 100, stresses / 1e6, 'o-', \n            linewidth=2, color=colors[idx % len(colors)],\n            label=f'{name} ({len(net.fibers)} fibers)')\n    \n    print(f'{name}: {len(net.fibers)} fibers, {len(strains)} points')\n\nax.set_xlabel('Strain (%)')\nax.set_ylabel('Stress (MPa)')\nax.set_title('Stress-Strain Curves (CPU FEM)')\nax.legend()\nax.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

### 4.2 Convergence Study / 收敛性研究\n\n验证 FEM 结果随网格密度的收敛性（h-refinement）。

In [ ]:
# ============================================================\n# Convergence study\n\ntest_net = random_2d(\n    num_fibers=40, fiber_length=5.0, box_size=(10, 10),\n    fiber_radius=0.05, material=material, seed=42\n)\n\nresult = validate_convergence(\n    network=test_net,\n    segment_counts=[2, 4, 8, 12, 16, 20],\n    axis=0,\n    strain=0.001,\n)\n\nprint(result.summary())\nprint()\nprint('Details:')\nfor key, value in result.details.items():\n    print(f'  {key}: {value}')\n\n# Plot convergence\nif 'segment_counts' in result.details:\n    fig, ax = plt.subplots(figsize=(10, 6))\n    \n    segs = result.details['segment_counts']\n    E_vals = result.details['effective_moduli']\n    \n    ax.plot(segs, E_vals, 'o-', linewidth=2, markersize=8)\n    ax.set_xlabel('Segments per fiber')\n    ax.set_ylabel('Effective modulus (Pa)')\n    ax.set_title('FEM Convergence (h-refinement)')\n    ax.grid(True, alpha=0.3)\n    \n    if len(E_vals) > 0:\n        ax.axhline(y=E_vals[-1], color='r', linestyle='--', \n                   label=f'Reference: {E_vals[-1]:.2e} Pa')\n        ax.legend()\n    \n    plt.tight_layout()\n    plt.show()

---\n## 5. Summary / 总结\n\n本教程展示了 FiberNet 的验证和强化功能：\n\n1. **TPMS 结构生成**: Gyroid, Diamond, Primitive 等三周期极小曲面点阵\n2. **FEM 验证**: 悬臂梁解析解、Patch Test、Gibson-Ashby 标度律\n3. **力学强化分析**: 密度效应、取向效应、应力-应变曲线\n4. **收敛性研究**: h-refinement 网格敏感性分析\n\n所有计算均在 CPU 上完成，无需 GPU 加速。\n\n**技术特点**：\n- 自研 FEM 实现（非开源 FEM 库）\n- scipy.sparse 高效稀疏求解\n- Euler-Bernoulli 梁单元理论\n- 支持 2D/3D 网络